In [1]:
import numpy as np
from pathlib import Path
import random
from transformers import ResNetModel
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2
import torch
import pandas as pd
import evaluate
from sklearn.model_selection import train_test_split


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

base_path = Path("dataset")
img_path = base_path / "Food Images"
cap_path = base_path / "Food Ingredients and Recipe Dataset with Image Name Mapping.csv"

assert base_path.exists()
assert img_path.exists()
assert cap_path.exists()

In [3]:
annotations_df = pd.read_csv(cap_path, index_col=0)

# Add .jpg extension to the 'Image_Name' column
annotations_df['Image_Name'] = annotations_df['Image_Name'].apply(lambda x: f'{x}.jpg')

# Retain only the 'Title' and 'Image_Name' columns
data = annotations_df[['Title', 'Image_Name']]

shuffled_indices = np.random.permutation(len(data))

# Step 2: Calculate split sizes
train_size = int(0.8 * len(annotations_df))
val_size = int(0.1 * len(annotations_df))
test_size = len(annotations_df) - train_size - val_size  # Remaining for test

# Step 3: Create the splits (train, validation, test)
train_indices = shuffled_indices[:train_size]
val_indices = shuffled_indices[train_size:train_size + val_size]
test_indices = shuffled_indices[train_size + val_size:]

# Step 4: Create partitions in the format expected by the Data class
partitions = {
    'train': train_indices,
    'val': val_indices,
    'test': test_indices
}

print(f"Train size: {(partitions['train'])}")
print(f"Validation size: {(partitions['val'])}")
print(f"Test size: {(partitions['test'])}")

Train size: [10023 10405  3171 ... 13244  1503  9469]
Validation size: [10526  6384  2379 ...  1636  1588  7154]
Test size: [ 1860 11279 13052 ... 10063  3406  9600]


In [18]:
data



,Title,Image_Name
0,Miso-Butter Roast Chicken With Acorn Squash Pa...,miso-butter-roast-chicken-acorn-squash-panzane...
1,Crispy Salt and Pepper Potatoes,crispy-salt-and-pepper-potatoes-dan-kluger.jpg
2,Thanksgiving Mac and Cheese,thanksgiving-mac-and-cheese-erick-williams.jpg
3,Italian Sausage and Bread Stuffing,italian-sausage-and-bread-stuffing-240559.jpg
4,Newton's Law,newtons-law-apple-bourbon-cocktail.jpg
...,...,...
13496,Brownie Pudding Cake,brownie-pudding-cake-14408.jpg
13497,Israeli Couscous with Roasted Butternut Squash...,israeli-couscous-with-roasted-butternut-squash...
13498,Rice with Soy-Glazed Bonito Flakes and Sesame ...,rice-with-soy-glazed-bonito-flakes-and-sesame-...
13499,Spanakopita,spanakopita-107344.jpg


In [7]:
chars = ['<SOS>', '<EOS>', '<PAD>', ' ', '!', '"', '#', '&', "  '", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
NUM_CHAR = len(chars)
idx2char = {k: v for k, v in enumerate(chars)}
char2idx = {v: k for k, v in enumerate(chars)}
print(chars[30], chars[34], chars[35], chars[33])
TEXT_MAX_LEN = 201
print(NUM_CHAR)

C G H F
80


In [12]:
class Data(Dataset):
    def __init__(self, data, partition):
        self.data = data
        self.partition = partition
        self.num_captions = 5
        self.max_len = TEXT_MAX_LEN
        self.img_proc = torch.nn.Sequential(
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Resize((224, 224), antialias=True),
            v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),)

    def __len__(self):
        return len(self.partition)
    
    def __getitem__(self, idx):
        real_idx = self.num_captions*self.partition[idx]
        item = self.data.iloc[real_idx: real_idx+self.num_captions]
        ## image processing
        img_name = item.image.reset_index(drop=True)[0]
        img = Image.open(f'{img_path}{img_name}').convert('RGB')
        img = self.img_proc(img)
    
        ## caption processing
        caption = item.caption.reset_index(drop=True)[random.choice(list(range(self.num_captions)))]
        cap_list = list(caption)
        final_list = [chars[0]]
        final_list.extend(cap_list)
        final_list.extend([chars[1]])
        gap = self.max_len - len(final_list)
        final_list.extend([chars[2]]*gap)
        cap_idx = [char2idx[i] for i in final_list]
        return img, cap_idx

In [34]:
class Data2(Dataset):
    def __init__(self, data, partition):
        self.data = data
        self.partition = partition
        self.max_len = TEXT_MAX_LEN
        self.img_proc = torch.nn.Sequential(
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Resize((224, 224), antialias=True),
            v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        )

    def __len__(self):
        return len(self.partition)
    
    def __getitem__(self, idx):
        # Get the index of the image-caption pair
        real_idx = self.partition[idx]  # Directly use partition for single captions

        # Retrieve the item (dish) from the data
        item = self.data.iloc[real_idx]
        
        ## image processing
        img_name = item.Image_Name  # Single image name per dish
        img = Image.open(f'{img_path}/{img_name}').convert('RGB')
        img = self.img_proc(img)
    
        ## caption processing
        caption = item.Title  # Single caption for the image
        cap_list = list(caption)  # Convert caption to a list of characters
        final_list = [chars[0]]  # Add <SOS> token at the start
        final_list.extend(cap_list)  # Add the caption characters
        final_list.extend([chars[1]])  # Add <EOS> token at the end
        
        # Pad the caption to max_len
        gap = self.max_len - len(final_list)
        final_list.extend([chars[2]] * gap)  # Add <PAD> tokens as needed

        # Convert characters to indices using char2idx
        cap_idx = [char2idx[i] for i in final_list]
        
        return img, cap_idx

In [13]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = ResNetModel.from_pretrained('microsoft/resnet-18').to(DEVICE)
        self.gru = nn.GRU(512, 512, num_layers=1)
        self.proj = nn.Linear(512, NUM_CHAR)
        self.embed = nn.Embedding(NUM_CHAR, 512)

    def forward(self, img):
        batch_size = img.shape[0]
        feat = self.resnet(img)
        feat = feat.pooler_output.squeeze(-1).squeeze(-1).unsqueeze(0) # 1, batch, 512
        start = torch.tensor(char2idx['<SOS>']).to(DEVICE)
        start_embed = self.embed(start) # 512
        start_embeds = start_embed.repeat(batch_size, 1).unsqueeze(0) # 1, batch, 512
        inp = start_embeds
        hidden = feat
        for t in range(TEXT_MAX_LEN-1): # rm <SOS>
            out, hidden = self.gru(inp, hidden)
            inp = torch.cat((inp, out[-1:]), dim=0) # N, batch, 512
    
        res = inp.permute(1, 0, 2) # batch, seq, 512
        res = self.proj(res) # batch, seq, 80
        res = res.permute(0, 2, 1) # batch, 80, seq
        return res

In [ ]:
'''A simple example to calculate loss of a single batch (size 2)'''

dataset = Data2(data, partitions['train'])
img1, caption1 = next(iter(dataset))
img2, caption2 = next(iter(dataset))
print(caption1)
caption1 = torch.tensor(caption1)
caption2 = torch.tensor(caption2)
img = torch.cat((img1.unsqueeze(0), img2.unsqueeze(0)))
caption = torch.cat((caption1.unsqueeze(0), caption2.unsqueeze(0)))
img, caption = img.to(DEVICE), caption.to(DEVICE)
print(caption.shape)
model = Model().to(DEVICE)
pred = model(img)
print(pred.shape)
crit = nn.CrossEntropyLoss()
loss = crit(pred, caption)

print(loss)



[0, 46, 69, 62, 56, 58, 57, 3, 34, 65, 54, 79, 58, 57, 3, 30, 54, 71, 71, 68, 73, 72, 3, 76, 62, 73, 61, 3, 46, 61, 58, 71, 71, 78, 3, 7, 3, 30, 62, 73, 71, 74, 72, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
torch.Size([2, 201])
torch.Size([2, 80, 201])


In [44]:
'''metrics'''
bleu = evaluate.load('bleu')
meteor = evaluate.load('meteor')
rouge = evaluate.load('rouge')

reference = [['A child in a pink dress is climbing up a set of stairs in an entry way .', 'A girl going into a wooden building .']]
prediction = ['A girl goes into a wooden building .']

res_b = bleu.compute(predictions=prediction, references=reference)
res_r = rouge.compute(predictions=prediction, references=reference)
res_m = meteor.compute(predictions=prediction, references=reference)

res_b, res_r, res_m

[nltk_data] Downloading package wordnet to /ghome/c5mcv07/nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /ghome/c5mcv07/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /ghome/c5mcv07/nltk_data...


({'bleu': 0.5946035575013605,
  'precisions': [0.875, 0.7142857142857143, 0.5, 0.4],
  'brevity_penalty': 1.0,
  'length_ratio': 1.0,
  'translation_length': 8,
  'reference_length': 8},
 {'rouge1': np.float64(0.8571428571428571),
  'rouge2': np.float64(0.6666666666666666),
  'rougeL': np.float64(0.8571428571428571),
  'rougeLsum': np.float64(0.8571428571428571)},
 {'meteor': np.float64(0.864795918367347)})

In [45]:
ref = [['A child is running in the campus']]
pred1 = ['A child is running']

res_b = bleu.compute(predictions=pred1, references=ref)
res_r = rouge.compute(predictions=pred1, references=ref)
res_m = meteor.compute(predictions=pred1, references=ref)

res_b, res_r, res_m

({'bleu': 0.4723665527410147,
  'precisions': [1.0, 1.0, 1.0, 1.0],
  'brevity_penalty': 0.4723665527410147,
  'length_ratio': 0.5714285714285714,
  'translation_length': 4,
  'reference_length': 7},
 {'rouge1': np.float64(0.7272727272727273),
  'rouge2': np.float64(0.6666666666666666),
  'rougeL': np.float64(0.7272727272727273),
  'rougeLsum': np.float64(0.7272727272727273)},
 {'meteor': np.float64(0.5923507462686567)})

In [46]:
ref = [['A child is running in the campus']]
pred1 = ['A child is']

res_b = bleu.compute(predictions=pred1, references=ref)
res_r = rouge.compute(predictions=pred1, references=ref)
res_m = meteor.compute(predictions=pred1, references=ref)

res_b, res_r, res_m

({'bleu': 0.0,
  'precisions': [1.0, 1.0, 1.0, 0.0],
  'brevity_penalty': 0.2635971381157267,
  'length_ratio': 0.42857142857142855,
  'translation_length': 3,
  'reference_length': 7},
 {'rouge1': np.float64(0.6),
  'rouge2': np.float64(0.5),
  'rougeL': np.float64(0.6),
  'rougeLsum': np.float64(0.6)},
 {'meteor': np.float64(0.44612794612794615)})

In [47]:
ref = [['A child is running in the campus']]
pred1 = ['A child campus']

res_b = bleu.compute(predictions=pred1, references=ref)
res_r = rouge.compute(predictions=pred1, references=ref)
res_m = meteor.compute(predictions=pred1, references=ref)
res_m_sin = meteor.compute(predictions=pred1, references=ref, gamma=0) # no penalty by setting gamma to 0

res_b, res_r, res_m, res_m_sin

({'bleu': 0.0,
  'precisions': [1.0, 0.5, 0.0, 0.0],
  'brevity_penalty': 0.2635971381157267,
  'length_ratio': 0.42857142857142855,
  'translation_length': 3,
  'reference_length': 7},
 {'rouge1': np.float64(0.6),
  'rouge2': np.float64(0.25),
  'rougeL': np.float64(0.6),
  'rougeLsum': np.float64(0.6)},
 {'meteor': np.float64(0.3872053872053872)},
 {'meteor': np.float64(0.45454545454545453)})

Final metric we use for challenge 3: BLEU1, BLEU2, ROUGE-L, METEOR

In [48]:
ref = [['A child is running in the campus']]
pred1 = ['A child campus']

bleu1 = bleu.compute(predictions=pred1, references=ref, max_order=1)
bleu2 = bleu.compute(predictions=pred1, references=ref, max_order=2)
res_r = rouge.compute(predictions=pred1, references=ref)
res_m = meteor.compute(predictions=pred1, references=ref)

f"BLEU-1:{bleu1['bleu']*100:.1f}%, BLEU2:{bleu2['bleu']*100:.1f}%, ROUGE-L:{res_r['rougeL']*100:.1f}%, METEOR:{res_m['meteor']*100:.1f}%"

'BLEU-1:26.4%, BLEU2:18.6%, ROUGE-L:60.0%, METEOR:38.7%'

Now it is your turn! Try to finish the code below to run the train function

In [ ]:
def train(EPOCHS):
    data_train = Data(data, partitions['train'])
    data_valid = Data(data, partitions['valid'])
    data_test = Data(data, partitions['test'])
    dataloader_train = DataLoader(data_train, batch_size=32, shuffle=True, num_workers=4)
    dataloader_valid = DataLoader(data_valid, batch_size=32, shuffle=False, num_workers=4)
    dataloader_test = DataLoader(data_test, batch_size=32, shuffle=False, num_workers=4)
    model = Model().to(DEVICE)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()
    metric = Metric()
    for epoch in range(EPOCHS):
        loss, res = train_one_epoch(model, optimizer, crit, metric, dataloader_train)
        print(f'train loss: {loss:.2f}, metric: {res:.2f}, epoch: {epoch}')
        loss_v, res_v = eval_epoch(model, crit, metric, dataloader_valid)
        print(f'valid loss: {loss:.2f}, metric: {res:.2f}')
    loss_t, res_t = eval_epoch(model, crit, metric, dataloader_test)
    print(f'test loss: {loss:.2f}, metric: {res:.2f}')
    
def train_one_epoch(model, optimizer, crit, metric, dataloader):
    model.train()  # Set model to training mode
    running_loss = 0.0
    running_metric = 0.0
    for img, caption in dataloader:
        # Move data to the proper device
        img, caption = img.to(DEVICE), caption.to(DEVICE)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        pred, _ = model(img)  # Assuming the model returns predictions (logits)
        
        # Compute loss
        loss = crit(pred, caption)
        
        # Backpropagate and update the model
        loss.backward()
        optimizer.step()
        
        # Compute evaluation metric (e.g., accuracy or BLEU score)
        # Assume 'metric' is an object that can compute the metric
        res = metric(pred, caption)  # This would depend on your implementation of Metric
        
        # Accumulate loss and metric
        running_loss += loss.item()
        running_metric += res
    
    # Average loss and metric
    avg_loss = running_loss / len(dataloader)
    avg_metric = running_metric / len(dataloader)
    
    return avg_loss, avg_metric


def eval_epoch(model, crit, metric, dataloader):
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    running_metric = 0.0
    with torch.no_grad():  # No gradient calculation during evaluation
        for img, caption in dataloader:
            # Move data to the proper device
            img, caption = img.to(DEVICE), caption.to(DEVICE)
            
            # Forward pass
            pred, _ = model(img)  # Assuming the model returns predictions (logits)
            
            # Compute loss
            loss = crit(pred, caption)
            
            # Compute evaluation metric (e.g., accuracy or BLEU score)
            res = metric(pred, caption)  # This would depend on your implementation of Metric
            
            # Accumulate loss and metric
            running_loss += loss.item()
            running_metric += res
    
    # Average loss and metric
    avg_loss = running_loss / len(dataloader)
    avg_metric = running_metric / len(dataloader)
    
    return avg_loss, avg_metric

